# Modelagem probabilística

Este notebook apresenta somente a estimação por classe e os diagnósticos de distribuição da modelagem probabilística. Os ajustes e fórmulas reutilizáveis estão em `src/distributions.py`.

- `age`: Normal com média e variância MLE (`ddof=0`).
- `duration`: Gamma com forma e escala MLE, localização fixa em zero; Exponencial apenas como comparação.
- `marital`: categórica com domínio congelado e Laplace (`alpha=1`).
- Priors: frequências empíricas de classe, sem suavização.

O split existente é realizado uma única vez. Somente suas linhas de treino são utilizadas abaixo. Classificadores, posterior e avaliação final pertencem aos módulos consumidores.

In [1]:
from pathlib import Path
import sys

# Funciona com o kernel iniciado na raiz do projeto ou em notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "src" / "config.py").is_file():
    project_root = project_root.parent
if not (project_root / "src" / "config.py").is_file():
    raise RuntimeError("Inicie o kernel na raiz do projeto ou em notebooks/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import DATA_PATH, CLASS_ORDER, MARITAL_CATEGORIES, LAPLACE_ALPHA
from src.data import load_bank_data, validate_raw_data, prepare_model_frame, make_stratified_split
from src.distributions import (
    fit_class_priors, fit_gaussian_mle, gaussian_logpdf, fit_gamma_mle,
    fit_exponential_mle, fit_categorical, compare_duration_distributions,
)

pd.set_option("display.precision", 6)

In [2]:
data_path = project_root / DATA_PATH
raw = load_bank_data(data_path)
validate_raw_data(raw, path=data_path)
X, y = prepare_model_frame(raw)
split = make_stratified_split(X, y)
X_train, y_train = split.X_train, split.y_train
del raw, X, y, split

assert X_train.index.equals(y_train.index)
priors = fit_class_priors(y_train)
prior_table = pd.DataFrame([
    {"class": c, "n_train": int((y_train == c).sum()), "prior": priors[c]}
    for c in CLASS_ORDER
])
display(prior_table)

,class,n_train,prior
0,0,3199,0.884679
1,1,417,0.115321


## Parâmetros estimados exclusivamente no treino

Cada ajuste abaixo recebe apenas as linhas da classe correspondente. `variance` é a variância MLE; `std_mle` é sua raiz quadrada, exibida para comparação com a especificação. A taxa Exponencial é um parâmetro comparativo.

O SciPy resolve numericamente o MLE Gamma com `gamma.fit(values, floc=0)`. A implementação valida forma e escala finitas e positivas, localização exatamente zero, média `shape * scale` próxima à média empírica e log-densidades finitas em todo o treino. A log-densidade é escrita explicitamente usando `scipy.special.gammaln`. Combinação de evidências e decisão serão implementadas pela dupla nos módulos de classificação.

In [3]:
parameter_rows = []
categorical_rows = []
diagnostic_tables = []

for c in CLASS_ORDER:
    training_class = X_train.loc[y_train == c]
    age_values = training_class["age"].to_numpy()
    duration_values = training_class["duration"].to_numpy()

    gaussian = fit_gaussian_mle(age_values)
    gamma = fit_gamma_mle(duration_values)
    exponential = fit_exponential_mle(duration_values)
    marital = fit_categorical(training_class["marital"], MARITAL_CATEGORIES, LAPLACE_ALPHA)

    parameter_rows.append({
        "class": c, "age_mean": gaussian.mean, "age_variance": gaussian.variance,
        "age_std_mle": np.sqrt(gaussian.variance),
        "duration_shape": gamma.shape, "duration_scale": gamma.scale,
        "duration_mean_empirical": duration_values.mean(),
        "duration_mean_gamma": gamma.shape * gamma.scale,
        "duration_exponential_rate": exponential.rate,
    })

    counts = training_class["marital"].value_counts().reindex(MARITAL_CATEGORIES, fill_value=0)
    for category in MARITAL_CATEGORIES:
        categorical_rows.append({
            "class": c, "category": category, "count": int(counts[category]),
            "frequency_unsmoothed": counts[category] / len(training_class),
            "probability_laplace": marital[category],
        })
    diagnostic_tables.append(compare_duration_distributions(duration_values).assign(**{"class": c}))

parameter_table = pd.DataFrame(parameter_rows)
categorical_table = pd.DataFrame(categorical_rows)
duration_comparison = pd.concat(diagnostic_tables, ignore_index=True)
display(parameter_table)

,class,age_mean,age_variance,age_std_mle,duration_shape,duration_scale,duration_mean_empirical,duration_mean_gamma,duration_exponential_rate
0,0,40.871835,101.245218,10.062068,1.528535,147.447809,225.379181,225.379181,0.004437
1,1,42.364508,170.696870,13.065101,2.252761,247.819513,558.278177,558.278177,0.001791


## Frequências categóricas e suavização

A tabela exibe as contagens, as frequências sem suavização e as probabilidades de Laplace para as três categorias. A soma suavizada de cada classe deve ser aproximadamente 1. Uma categoria válida ausente no treino ainda recebe probabilidade positiva. Categorias desconhecidas geram erro, sem mapeamento silencioso para `<UNK>`.

In [4]:
display(categorical_table)
assert np.allclose(
    categorical_table.groupby("class")["probability_laplace"].sum().to_numpy(), 1.0
)

# Exemplo didático: divorced está ausente; o produto sem Laplace seria anulado.
example = pd.Series(["married", "married", "single"])
unsmoothed = example.value_counts(normalize=True).reindex(MARITAL_CATEGORIES, fill_value=0)
smoothed = fit_categorical(example)
display(pd.DataFrame({
    "frequency_unsmoothed": unsmoothed,
    "probability_laplace": pd.Series(smoothed),
}))

,class,category,count,frequency_unsmoothed,probability_laplace
0,0,divorced,366,0.114411,0.114616
1,0,married,2003,0.626133,0.625859
2,0,single,830,0.259456,0.259525
3,1,divorced,64,0.153477,0.154762
4,1,married,220,0.527578,0.526190
5,1,single,133,0.318945,0.319048


,frequency_unsmoothed,probability_laplace
divorced,0.000000,0.166667
married,0.666667,0.500000
single,0.333333,0.333333


## Gamma × Exponencial em duration

AIC usa `2q - 2ℓ`, com `q=2` para Gamma e `q=1` para Exponencial. A localização é fixa e não é um parâmetro livre. A Normal tem `q=2`, mas seu AIC para idade não deve ser comparado ao AIC de duração.

Menores valores de AIC e da estatística KS indicam melhor ajuste relativo na mesma classe e atributo. Como os parâmetros foram estimados nos próprios dados, o KS é um diagnóstico relativo; não usamos seu p-valor como prova isolada de aderência.

In [5]:
display(duration_comparison[["class", "distribution", "q", "log_likelihood", "aic", "ks"]])
for c in CLASS_ORDER:
    class_comparison = duration_comparison.loc[duration_comparison["class"] == c].set_index("distribution")
    for metric in ("aic", "ks"):
        gamma_better = class_comparison.loc["Gamma", metric] < class_comparison.loc["Exponencial", metric]
        print(f"Classe {c}: Gamma tem menor {metric.upper()}? {bool(gamma_better)}")

,class,distribution,q,log_likelihood,aic,ks
0,0,Exponencial,1,-20530.491760,41062.983520,0.117250
1,0,Gamma,2,-20375.827910,40755.655821,0.045733
2,1,Exponencial,1,-3054.465521,6110.931042,0.180231
3,1,Gamma,2,-2991.026841,5986.053682,0.052479


Classe 0: Gamma tem menor AIC? True
Classe 0: Gamma tem menor KS? True
Classe 1: Gamma tem menor AIC? True
Classe 1: Gamma tem menor KS? True


## Exemplo numérico manual da Normal

Para os valores fictícios `[1, 2, 3]`, a média é 2 e a variância MLE é `(1 + 0 + 1) / 3 = 2/3`. No ponto `x=2`, a log-densidade é `-0,5 × log(4π/3) ≈ -0,7162059792`. Esse exemplo não participa dos ajustes de treino.

In [6]:
example_params = fit_gaussian_mle(np.array([1.0, 2.0, 3.0]))
example_log_density = gaussian_logpdf(np.array([2.0]), example_params)[0]
assert np.isclose(example_params.mean, 2.0)
assert np.isclose(example_params.variance, 2 / 3)
assert np.isclose(example_log_density, -0.7162059791505905)
display(pd.DataFrame([{
    "mean": example_params.mean, "variance_mle": example_params.variance,
    "x": 2.0, "log_density": example_log_density,
}]))

,mean,variance_mle,x,log_density
0,2.0,0.666667,2.0,-0.716206


## Interpretação e limites

Os diagnósticos acima sustentam a Gamma como distribuição principal de duração para as duas classes. A Exponencial mantém a forma decrescente e o coeficiente de variação unitário, restrições que limitam seu ajuste. Esta comparação não mede desempenho de classificação.

Idade é discreta, limitada e pode conter subpopulações; a Normal é uma aproximação didática. Os valores das funções contínuas são **densidades**, não probabilidades pontuais. As frequências de classe são **priors** e os valores de `marital` são **probabilidades categóricas condicionais**. A posterior exige normalização entre classes; o produto de prior e verossimilhanças usado para decisão não é, por si só, uma posterior normalizada.

A implementação trabalha diretamente em log e não substitui resultados não finitos silenciosamente. O piso de variância só atua se a estimativa for nula na precisão numérica.

Referências: modelagem probabilística, [SciPy Gamma](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gamma.html) e [gammaln](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.gammaln.html).